In [1]:
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

con = duckdb.connect("data/energy.duckdb")

# Recreate the simulated price data
np.random.seed(42)
dates = pd.date_range("2025-01-01", "2025-12-31", freq="h")
base = 65 + 15 * np.sin(np.arange(len(dates)) * 2 * np.pi / 24)
noise = np.random.normal(0, 10, len(dates))
seasonal = 10 * np.sin(np.arange(len(dates)) * 2 * np.pi / (365*24))
prices = base + noise + seasonal
prices = np.clip(prices, 0, 300)

df = pd.DataFrame({"timestamp": dates, "price_eur_mwh": prices})

# Load into DuckDB
con.execute("DROP TABLE IF EXISTS da_prices")
con.execute("CREATE TABLE da_prices AS SELECT * FROM df")

# Verify
print(con.execute("SELECT COUNT(*) FROM da_prices").fetchdf())

   count_star()
0          8737


In [2]:
#  Day 8 - SQL — CTEs and subqueries

# Common Table Expressions (CTEs) — the most important SQL pattern for analytics
# Find days where the daily average exceeded the monthly average

print(con.execute("""
    WITH daily_avg AS (
        SELECT
            CAST(timestamp AS DATE) as date,
            ROUND(AVG(price_eur_mwh), 2) as daily_price
        FROM da_prices
        GROUP BY CAST(timestamp AS DATE)
    ),
    monthly_avg AS (
        SELECT
            MONTH(date) as month,
            ROUND(AVG(daily_price), 2) as monthly_price
        FROM daily_avg
        GROUP BY MONTH(date)
    )
    SELECT
        d.date,
        d.daily_price,
        m.monthly_price,
        ROUND(d.daily_price - m.monthly_price, 2) as deviation
    FROM daily_avg d
    JOIN monthly_avg m ON MONTH(d.date) = m.month
    WHERE d.daily_price > m.monthly_price * 1.05
    ORDER BY deviation DESC
    LIMIT 15
""").fetchdf())

         date  daily_price  monthly_price  deviation
0  2025-11-30        65.00          58.23       6.77
1  2025-06-03        73.31          67.07       6.24
2  2025-01-26        72.92          67.46       5.46
3  2025-12-29        67.50          62.64       4.86
4  2025-06-02        71.89          67.07       4.82
5  2025-04-07        79.68          74.87       4.81
6  2025-06-15        71.82          67.07       4.75
7  2025-09-10        59.52          55.04       4.48
8  2025-08-09        62.09          57.61       4.48
9  2025-06-04        71.39          67.07       4.32
10 2025-12-31        66.90          62.64       4.26
11 2025-04-23        79.06          74.87       4.19
12 2025-08-03        61.64          57.61       4.03
13 2025-11-28        62.11          58.23       3.88
14 2025-11-19        61.95          58.23       3.72


In [3]:
# Day 8 Exercise: Write a CTE that identifies "price spike hours" (price > 2x the daily average) and counts how many occur per month.

print(con.execute("""
    WITH daily_avg AS (
        SELECT
            CAST(timestamp AS DATE) as date,
            ROUND(AVG(price_eur_mwh), 2) as daily_price
        FROM da_prices
        GROUP BY CAST(timestamp AS DATE)
    ),
    spike_hours AS (
        SELECT
            p.timestamp,
            p.price_eur_mwh,
            d.daily_price,
            ROUND(p.price_eur_mwh / d.daily_price, 2) as price_ratio
        FROM da_prices p
        JOIN daily_avg d ON CAST(p.timestamp AS DATE) = d.date
        WHERE p.price_eur_mwh > d.daily_price * 2
    )
    SELECT
        MONTH(timestamp) as month,
        COUNT(*) as spike_hours,
        ROUND(AVG(price_eur_mwh), 2) as avg_spike_price,
        ROUND(AVG(price_ratio), 2) as avg_price_ratio,
        MAX(price_eur_mwh) as worst_spike
    FROM spike_hours
    GROUP BY MONTH(timestamp)
    ORDER BY month
""").fetchdf())

Empty DataFrame
Columns: [month, spike_hours, avg_spike_price, avg_price_ratio, worst_spike]
Index: []


In [4]:
# Day 9

# Window functions are essential for time-series analytics in SQL
# LAG/LEAD: compare each hour to previous/next

print(con.execute("""
SELECT
    timestamp,
    price_eur_mwh,
    LAG(price_eur_mwh, 1) OVER (ORDER BY timestamp) AS prev_hour,
    ROUND(
        price_eur_mwh - LAG(price_eur_mwh, 1) OVER (ORDER BY timestamp),
        2
    ) AS hour_change,
    RANK() OVER (
        PARTITION BY CAST(timestamp AS DATE)
        ORDER BY price_eur_mwh DESC
    ) AS rank_in_day
FROM da_prices
LIMIT 30
""").fetchdf())


# Running totals and moving averages

print(con.execute("""
SELECT
    timestamp,
    price_eur_mwh,
    ROUND(
        AVG(price_eur_mwh) OVER (
            ORDER BY timestamp
            ROWS BETWEEN 167 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS rolling_7d_avg,
    SUM(price_eur_mwh) OVER (
        PARTITION BY CAST(timestamp AS DATE)
        ORDER BY timestamp
    ) AS cumulative_daily
FROM da_prices
LIMIT 48
""").fetchdf())

             timestamp  price_eur_mwh  prev_hour  hour_change  rank_in_day
0  2025-01-08 11:00:00      97.364346  71.126686        26.24            1
1  2025-01-08 09:00:00      91.408083  79.379425        12.03            2
2  2025-01-08 07:00:00      89.012629  84.011701         5.00            3
3  2025-01-08 05:00:00      84.138083  78.449922         5.69            4
4  2025-01-08 06:00:00      84.011701  84.138083        -0.13            5
5  2025-01-08 08:00:00      79.379425  89.012629        -9.63            6
6  2025-01-08 04:00:00      78.449922  68.671938         9.78            7
7  2025-01-08 12:00:00      72.544155  97.364346       -24.82            8
8  2025-01-08 10:00:00      71.126686  91.408083       -20.28            9
9  2025-01-08 23:00:00      71.047385  54.393427        16.65           10
10 2025-01-08 03:00:00      68.671938  64.821176         3.85           11
11 2025-01-08 02:00:00      64.821176  62.554125         2.27           12
12 2025-01-08 00:00:00   

In [5]:
# Day 9 Exercise: For each day, find the hour with the highest price (use RANK or ROW_NUMBER). Return only rank=1 rows.

print(con.execute("""
    WITH ranked AS (
        SELECT
            CAST(timestamp AS DATE) AS date,
            timestamp,
            price_eur_mwh,
            RANK() OVER (
                PARTITION BY CAST(timestamp AS DATE)
                ORDER BY price_eur_mwh DESC
            ) AS rnk
        FROM da_prices
    )
    SELECT date, timestamp, ROUND(price_eur_mwh, 2) AS daily_high
    FROM ranked
    WHERE rnk = 1
    ORDER BY date
""").fetchdf())

          date           timestamp  daily_high
0   2025-01-01 2025-01-01 06:00:00       95.84
1   2025-01-02 2025-01-02 07:00:00       98.23
2   2025-01-03 2025-01-03 06:00:00       90.70
3   2025-01-04 2025-01-04 10:00:00       87.87
4   2025-01-05 2025-01-05 10:00:00       92.12
5   2025-01-06 2025-01-06 05:00:00      102.29
6   2025-01-07 2025-01-07 12:00:00       84.77
7   2025-01-08 2025-01-08 11:00:00       97.36
8   2025-01-09 2025-01-09 17:00:00       90.53
9   2025-01-10 2025-01-10 04:00:00      102.71
10  2025-01-11 2025-01-11 08:00:00       97.41
11  2025-01-12 2025-01-12 06:00:00       96.34
12  2025-01-13 2025-01-13 05:00:00       90.07
13  2025-01-14 2025-01-14 11:00:00       92.10
14  2025-01-15 2025-01-15 06:00:00       84.88
15  2025-01-16 2025-01-16 04:00:00       87.47
16  2025-01-17 2025-01-17 03:00:00       97.11
17  2025-01-18 2025-01-18 08:00:00       96.44
18  2025-01-19 2025-01-19 04:00:00       97.35
19  2025-01-20 2025-01-20 04:00:00      100.88
20  2025-01-2

In [6]:
# Day 12 Combine SQL + market knowledge

import duckdb
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

con = duckdb.connect("data/energy.duckdb")

# Table 1: market_participants
# A registry of every company active in the Slovenian electricity market.

con.execute("""
CREATE TABLE IF NOT EXISTS market_participants (
    participant_id INT PRIMARY KEY,  -- unique number for each participant
    name VARCHAR,                    -- company name
    role VARCHAR,                    -- 'BRP', 'supplier', 'producer', 'trader'
    balance_scheme_member BOOLEAN,   -- are they in the Balance Scheme? yes/no
    active_since DATE                -- when did they join?
)
""")

# Table 2: imbalance_settlement
# One row per 15-minute accounting interval.
# This is what Borzen calculates after the fact — comparing actual metered
# flows against the market plan, and attaching a price to any deviation.
con.execute("""
CREATE TABLE IF NOT EXISTS imbalance_settlement (
    settlement_period TIMESTAMP,      -- which 15-min window this row covers
    duration_minutes INT,             -- 15 or 60 (product granularity)
    system_imbalance_mw FLOAT,        -- how many MW off was the system? (+ or -)
    imbalance_price_eur_mwh FLOAT,    -- cost per MWh of being imbalanced
    direction VARCHAR                 -- was the system 'long' or 'short'?
)
""")

# Table 3: balancing_bids
# One row per bid submitted to the Platform for Balancing Energy.
# Generators and flexible consumers offer capacity here;
# ELES (TSO) decides which bids to activate to fix system imbalances.
con.execute("""
CREATE TABLE IF NOT EXISTS balancing_bids (
    bid_id INT,
    timestamp TIMESTAMP,
    product VARCHAR,        -- which reserve type: aFRR_up/down, mFRR_up/down
    volume_mw FLOAT,        -- how many MW they're offering
    price_eur_mwh FLOAT,    -- at what price
    activated BOOLEAN       -- did ELES actually call this bid?
)
""")

# ==============================================================================
# STEP 2: INSERT SAMPLE DATA USING NUMPY
# numpy.random lets us generate realistic fake numbers quickly.
# We simulate 50 settlement periods (= 50 x 15-minute windows = ~12.5 hours).
# ==============================================================================

np.random.seed(42)  # fixed seed = same "random" numbers every run
N = 50              # number of rows to generate

# --- market_participants: 10 real Slovenian market participants ---
participants = pd.DataFrame({
    "participant_id": range(1, 11),
    "name": [
        "HSE",           # Holding Slovenske Elektrarne – largest Slovenian producer
        "GEN energija",  # nuclear + hydro producer (Krško, Soča)
        "GEN-I",         # trading and supply arm of GEN group
        "Petrol",        # major supplier, retail energy
        "E3",            # supplier and trader
        "Energija plus", # retail supplier (Petrol subsidiary)
        "ELES",          # TSO – in the scheme as system operator
        "Borzen",        # Market Operator – head of the Balance Scheme
        "Javna razsvetljava", # smaller municipal supplier
        "IBEX Energy",   # regional trader
    ],
    # roles assigned to match real-world function of each company
    "role": [
        "producer", "producer", "trader", "supplier", "trader",
        "supplier", "BRP", "BRP", "supplier", "trader"
    ],
    "balance_scheme_member": [True] * 10,  # all active players are members
    "active_since": pd.date_range("2018-01-01", periods=10, freq="180D").date
})
con.execute("DELETE FROM market_participants")  # clear before re-inserting
con.executemany(
    "INSERT INTO market_participants VALUES (?, ?, ?, ?, ?)",
    participants.values.tolist()
)
 
# --- imbalance_settlement: 50 rows, one per 15-minute window ---
# Start at 2024-01-15 00:00, each row is 15 minutes later
start = datetime(2024, 1, 15, 0, 0)
periods = [start + timedelta(minutes=15 * i) for i in range(N)]
 
# system_imbalance_mw: normally distributed around 0 (system is roughly balanced)
# positive = system is long (too much generation), negative = short (too little)
imbalance_mw = np.random.normal(loc=0, scale=50, size=N)
 
# imbalance_price: base price ~60 EUR/MWh, spikes when imbalance is large
# abs() because price is always positive; we add noise on top
imbalance_price = np.maximum(
    60 + np.abs(imbalance_mw) * 0.4 + np.random.normal(0, 10, N),
    20  # floor at 20 EUR/MWh — imbalance prices are always positive
)
 
# direction is derived from the sign of imbalance_mw
# note: exact zero is theoretically possible with continuous random numbers but vanishingly rare
direction = ["long" if v > 0 else "short" if v < 0 else "balanced" for v in imbalance_mw]
 
settlement_data = pd.DataFrame({
    "settlement_period": periods,
    "duration_minutes": [15] * N,
    "system_imbalance_mw": imbalance_mw.round(2),
    "imbalance_price_eur_mwh": imbalance_price.round(2),
    "direction": direction
})
con.execute("DELETE FROM imbalance_settlement")
con.executemany(
    "INSERT INTO imbalance_settlement VALUES (?, ?, ?, ?, ?)",
    settlement_data.values.tolist()
)
 
# --- balancing_bids: 50 bids submitted to the balancing platform ---
products = ["aFRR_up", "aFRR_down", "mFRR_up", "mFRR_down"]
 
bids_data = pd.DataFrame({
    "bid_id": range(1, N + 1),
    "timestamp": periods,
    # randomly assign each bid to a reserve product
    # note: in reality _up bids activate when system is short, _down when long
    # here products are random for simplicity — not linked to system_imbalance direction
    "product": np.random.choice(products, size=N),
    # volume between 5 and 100 MW
    "volume_mw": np.random.uniform(5, 100, N).round(1),
    # price between 20 and 200 EUR/MWh (balancing bids span a wide range)
    "price_eur_mwh": np.random.uniform(20, 200, N).round(2),
    # ~30% of bids get activated by ELES
    "activated": np.random.choice([True, False], size=N, p=[0.3, 0.7])
})
con.execute("DELETE FROM balancing_bids")
con.executemany(
    "INSERT INTO balancing_bids VALUES (?, ?, ?, ?, ?, ?)",
    bids_data.values.tolist()
)
 
print("Tables created and populated.")
print(f"  market_participants: {con.execute('SELECT COUNT(*) FROM market_participants').fetchone()[0]} rows")
print(f"  imbalance_settlement: {con.execute('SELECT COUNT(*) FROM imbalance_settlement').fetchone()[0]} rows")
print(f"  balancing_bids: {con.execute('SELECT COUNT(*) FROM balancing_bids').fetchone()[0]} rows")

Tables created and populated.
  market_participants: 10 rows
  imbalance_settlement: 50 rows
  balancing_bids: 50 rows


In [7]:
# Day 12 Exercise: Exercise: Insert 50 rows of simulated imbalance settlement data (use numpy random). Write 3 queries:
# 1. average imbalance price by direction
# 2. hours with highest absolute imbalance
# 3. correlation between imbalance volume and price

# Query 1: Average imbalance price by direction (long vs short)
# This tells you whether the system tends to pay more when it's short or long.
print("\n--- Q1: Average imbalance price by direction ---")
print(con.execute("""
    SELECT
        direction,
        COUNT(*) as periods,
        ROUND(AVG(imbalance_price_eur_mwh), 2) as avg_price,
        ROUND(AVG(ABS(system_imbalance_mw)), 2) as avg_abs_imbalance_mw
    FROM imbalance_settlement
    GROUP BY direction
    ORDER BY direction
""").fetchdf())
 
# Query 2: Top 10 periods with the highest absolute imbalance
# Large absolute imbalances = the system was most stressed in these windows.
print("\n--- Q2: Hours with highest absolute imbalance ---")
print(con.execute("""
    SELECT
        settlement_period,
        system_imbalance_mw,
        direction,
        imbalance_price_eur_mwh
    FROM imbalance_settlement
    ORDER BY ABS(system_imbalance_mw) DESC
    LIMIT 10
""").fetchdf())
 
# Query 3: Correlation between imbalance volume and price
# We expect a positive correlation: bigger imbalance → higher price.
# CORR() is a built-in DuckDB aggregate function.
print("\n--- Q3: Correlation between imbalance volume and price ---")
print(con.execute("""
    SELECT
        ROUND(CORR(ABS(system_imbalance_mw), imbalance_price_eur_mwh), 4) as correlation
    FROM imbalance_settlement
""").fetchdf())


--- Q1: Average imbalance price by direction ---
  direction  periods  avg_price  avg_abs_imbalance_mw
0      long       20      75.09                 33.81
1     short       30      75.78                 41.33

--- Q2: Hours with highest absolute imbalance ---
    settlement_period  system_imbalance_mw direction  imbalance_price_eur_mwh
0 2024-01-15 09:15:00           -97.980003     short               102.480003
1 2024-01-15 03:15:00           -95.660004     short                86.300003
2 2024-01-15 07:45:00            92.610001      long               100.620003
3 2024-01-15 12:15:00           -88.150002     short                92.910004
4 2024-01-15 03:30:00           -86.250000     short               102.620003
5 2024-01-15 01:30:00            78.959999      long                83.190002
6 2024-01-15 00:45:00            76.150002      long                96.580002
7 2024-01-15 11:00:00           -73.930000     short                85.650002
8 2024-01-15 05:00:00            73